# MYSignVoice annotated dataset update

This notebook downloads a frozen Roboflow version, audits its split and 63-class ID order, measures the current deployed model on Validation, fine-tunes from that checkpoint, keeps the final Test split closed during tuning, and exports the accepted model to PyTorch, ONNX, and OpenVINO.

Read `docs/annotated_update_export_guide.md` first. Run the cells from top to bottom. Do not place a Roboflow API key directly in this notebook.


## 1. Install the tested packages


In [ ]:
%pip install -q "ultralytics==8.4.128" roboflow pyyaml pandas matplotlib openvino onnx onnxruntime


## 2. Confirm the GPU and connect private storage

In Colab, add `ROBOFLOW_API_KEY` in the Secrets panel and allow this notebook to access it. The key is never printed.


In [ ]:
import os
import platform
from getpass import getpass
from pathlib import Path

import torch
import ultralytics

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    BASE_DIR = Path("/content")
    BACKUP_ROOT = Path("/content/drive/MyDrive/TrafficSignProject/training_runs")
    api_key = userdata.get("ROBOFLOW_API_KEY")
else:
    BASE_DIR = Path("/workspace")
    BACKUP_ROOT = BASE_DIR / "mysignvoice_artifacts"
    api_key = os.environ.get("ROBOFLOW_API_KEY") or getpass("Roboflow private API key: ")

assert api_key, "ROBOFLOW_API_KEY is missing."
assert torch.cuda.is_available(), "A CUDA GPU is required. In Colab select a T4 GPU runtime."
os.environ["ROBOFLOW_API_KEY"] = api_key
BACKUP_ROOT.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version())
print("Ultralytics:", ultralytics.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("Persistent output:", BACKUP_ROOT)


## 3. Enter the frozen Roboflow version and split approval

Change `VERSION` to the number shown by Roboflow. On the first run, leave `EXPECTED_SPLIT_IMAGES = None` and `SPLIT_APPROVED = False`. The audit cell prints the real counts. Copy those counts here only after comparing them with the Roboflow version page, then set `SPLIT_APPROVED = True`.


In [ ]:
WORKSPACE = "kendrewlim-yahoo-com"
PROJECT = "mysignvoice-49-signs"
VERSION = 3  # Change this if Roboflow shows another version number.

# Example after auditing: {"train": 7000, "val": 1800, "test": 900}
EXPECTED_SPLIT_IMAGES = None
SPLIT_APPROVED = False
RUN_FINAL_TEST = False

BASE_MODEL_PATH = BASE_DIR / "best.pt"
EPOCHS = 100
IMAGE_SIZE = 640
BATCH_SIZE = 16
PATIENCE = 20
WORKERS = 8
CLASS_WEIGHT_POWER = 0.25

DATASET_DIR = BASE_DIR / f"mysignvoice_rf_v{VERSION}"
RUN_ROOT = BASE_DIR / "mysignvoice_runs"
RUN_NAME = f"yolo26s_63class_rf_v{VERSION}_annotated_update"
RUN_DIR = RUN_ROOT / RUN_NAME
EVAL_ROOT = BASE_DIR / "mysignvoice_evaluations"
REPORT_DIR = BASE_DIR / f"mysignvoice_report_rf_v{VERSION}"
for folder in (RUN_ROOT, EVAL_ROOT, REPORT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
print("Roboflow version:", VERSION)
print("Expected split:", EXPECTED_SPLIT_IMAGES)
print("Split approved:", SPLIT_APPROVED)


## 4. Upload and verify the current deployed Version 2 checkpoint

Upload this repository's `models/best.pt`. The notebook fine-tunes from it and uses its 63 output IDs as the source of truth.


In [ ]:
if IN_COLAB and not BASE_MODEL_PATH.is_file():
    from google.colab import files
    print("Upload models/best.pt now.")
    uploaded = files.upload()
    assert len(uploaded) == 1, "Upload exactly one best.pt file."
    uploaded_name = next(iter(uploaded))
    Path(uploaded_name).replace(BASE_MODEL_PATH)

from ultralytics import YOLO
assert BASE_MODEL_PATH.is_file(), f"Missing checkpoint: {BASE_MODEL_PATH}"
base_model = YOLO(str(BASE_MODEL_PATH), task="detect")
EXPECTED_CLASSES = base_model.names
if isinstance(EXPECTED_CLASSES, dict):
    EXPECTED_CLASSES = [EXPECTED_CLASSES[i] for i in range(len(EXPECTED_CLASSES))]
assert len(EXPECTED_CLASSES) == 63 and len(set(EXPECTED_CLASSES)) == 63, (
    "The uploaded checkpoint is not the deployed 63-class MYSignVoice model."
)
print("PASS: deployed checkpoint contains 63 unique class IDs.")
for class_id, class_name in enumerate(EXPECTED_CLASSES):
    print(f"{class_id:>2}: {class_name}")


## 5. Export and download the frozen Roboflow version

The Python package creates the compatible Ultralytics export and downloads it directly. No manual ZIP transfer is required.


In [ ]:
from roboflow import Roboflow

existing_yaml = DATASET_DIR / "data.yaml"
if existing_yaml.is_file():
    DATASET_ROOT = DATASET_DIR
    print("Reusing downloaded dataset:", DATASET_ROOT)
else:
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    rf_version = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION)
    try:
        dataset = rf_version.download(model_format="yolo26", location=str(DATASET_DIR))
    except Exception as yolo26_error:
        print("YOLO26 export was unavailable; using compatible YOLOv8 layout.")
        dataset = rf_version.download(model_format="yolov8", location=str(DATASET_DIR))
    DATASET_ROOT = Path(dataset.location)

DATA_YAML = DATASET_ROOT / "data.yaml"
assert DATA_YAML.is_file(), f"Missing data.yaml: {DATA_YAML}"
print("Dataset root:", DATASET_ROOT)
print("Dataset YAML:", DATA_YAML)


## 6. Audit class IDs, labels, split counts, and exact duplicates

This cell does not prove that neighbouring camera frames stayed together; verify that from Roboflow batches/tags. It does stop invalid labels, wrong class order, missing splits, empty training classes, and exact duplicate image files across splits.


In [ ]:
from collections import Counter, defaultdict
import hashlib
import json

import pandas as pd
import yaml

with DATA_YAML.open("r", encoding="utf-8") as stream:
    data_config = yaml.safe_load(stream)

exported_names = data_config.get("names", [])
if isinstance(exported_names, dict):
    exported_names = [
        exported_names[i] if i in exported_names else exported_names[str(i)]
        for i in range(len(exported_names))
    ]
NAME_CORRECTIONS = {
    "pass-obstacles-on-either-side": "pass-obstacle-on-either-side",
    "winding_road_warning": "winding-road-warning",
}
exported_names = [NAME_CORRECTIONS.get(name, name) for name in exported_names]
assert exported_names == EXPECTED_CLASSES, (
    "STOP: Roboflow class names or ID order differ from the deployed model.\n"
    f"Expected: {EXPECTED_CLASSES}\nExported: {exported_names}"
)

def resolve_split(yaml_key, fallback_folder):
    candidates = []
    raw_path = data_config.get(yaml_key)
    if raw_path:
        raw_path = Path(raw_path)
        candidates.append(raw_path if raw_path.is_absolute() else (DATA_YAML.parent / raw_path).resolve())
    candidates.append((DATASET_ROOT / fallback_folder / "images").resolve())
    image_dir = next((path for path in candidates if path.is_dir()), None)
    assert image_dir is not None, f"Cannot find {yaml_key} images. Tried: {candidates}"
    return image_dir

split_image_dirs = {
    "train": resolve_split("train", "train"),
    "val": resolve_split("val", "valid"),
    "test": resolve_split("test", "test"),
}
image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
split_rows = []
box_counts = {split: Counter() for split in split_image_dirs}
invalid_labels = []
image_hash_splits = defaultdict(set)

for split_name, image_dir in split_image_dirs.items():
    label_dir = image_dir.parent / "labels"
    images = sorted(path for path in image_dir.iterdir() if path.suffix.lower() in image_extensions)
    assert images, f"The {split_name} split has no images."
    images_with_boxes = 0
    total_boxes = 0
    for image_path in images:
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        image_hash_splits[digest].add(split_name)
        label_file = label_dir / f"{image_path.stem}.txt"
        image_box_count = 0
        if label_file.is_file():
            for line_number, line in enumerate(label_file.read_text(encoding="utf-8").splitlines(), 1):
                if not line.strip():
                    continue
                parts = line.split()
                try:
                    class_id = int(parts[0])
                    coordinates = [float(value) for value in parts[1:]]
                    assert len(coordinates) == 4
                    assert 0 <= class_id < len(EXPECTED_CLASSES)
                    assert all(0.0 <= value <= 1.0 for value in coordinates)
                    assert coordinates[2] > 0.0 and coordinates[3] > 0.0
                except (ValueError, AssertionError):
                    invalid_labels.append(f"{label_file}:{line_number}: {line}")
                    continue
                image_box_count += 1
                total_boxes += 1
                box_counts[split_name][class_id] += 1
        if image_box_count:
            images_with_boxes += 1
    split_rows.append({
        "split": split_name,
        "images": len(images),
        "images_with_boxes": images_with_boxes,
        "negative_images": len(images) - images_with_boxes,
        "boxes": total_boxes,
    })

assert not invalid_labels, "Invalid labels:\n" + "\n".join(invalid_labels[:20])
duplicate_splits = {digest: sorted(splits) for digest, splits in image_hash_splits.items() if len(splits) > 1}
assert not duplicate_splits, (
    "STOP: exact duplicate image content exists across splits: "
    + json.dumps(dict(list(duplicate_splits.items())[:10]), indent=2)
)

split_summary = pd.DataFrame(split_rows).set_index("split")
ACTUAL_SPLIT_IMAGES = {name: int(split_summary.loc[name, "images"]) for name in split_image_dirs}
missing_train_classes = [EXPECTED_CLASSES[i] for i in range(63) if box_counts["train"][i] == 0]
assert not missing_train_classes, f"STOP: classes missing from Train: {missing_train_classes}"

class_rows = []
for class_id, class_name in enumerate(EXPECTED_CLASSES):
    class_rows.append({
        "class_id": class_id,
        "class_name": class_name,
        "train_boxes": box_counts["train"][class_id],
        "val_boxes": box_counts["val"][class_id],
        "test_boxes": box_counts["test"][class_id],
    })
class_distribution = pd.DataFrame(class_rows)
split_summary.to_csv(REPORT_DIR / "dataset_split_summary.csv")
class_distribution.to_csv(REPORT_DIR / "class_distribution.csv", index=False)
(REPORT_DIR / "split_counts.json").write_text(json.dumps(ACTUAL_SPLIT_IMAGES, indent=2), encoding="utf-8")

display(split_summary)
display(class_distribution.sort_values(["train_boxes", "class_id"]).head(20))
print("COPY THIS INTO EXPECTED_SPLIT_IMAGES:")
print(ACTUAL_SPLIT_IMAGES)
if EXPECTED_SPLIT_IMAGES is not None:
    assert ACTUAL_SPLIT_IMAGES == EXPECTED_SPLIT_IMAGES, (
        f"Split changed. Expected {EXPECTED_SPLIT_IMAGES}, got {ACTUAL_SPLIT_IMAGES}."
    )
if not SPLIT_APPROVED:
    print("NOT APPROVED: inspect Roboflow, enter the expected counts, set SPLIT_APPROVED=True, and rerun.")
else:
    assert EXPECTED_SPLIT_IMAGES is not None, "Enter expected counts before approval."
    print("PASS: dataset audit and frozen split approval completed.")


## 7. Measure the deployed model on the new Validation split

This is the fair baseline. The Test split remains unopened.


In [ ]:
assert SPLIT_APPROVED and EXPECTED_SPLIT_IMAGES == ACTUAL_SPLIT_IMAGES, "Approve the audited split first."
import json
import numpy as np

baseline_metrics = base_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=0, project=str(EVAL_ROOT),
    name=f"deployed_model_on_rf_v{VERSION}_validation", exist_ok=True, plots=True,
)
BASELINE_SUMMARY = {
    "model": "deployed Version 2 best.pt",
    "precision": float(baseline_metrics.box.mp),
    "recall": float(baseline_metrics.box.mr),
    "mAP50": float(baseline_metrics.box.map50),
    "mAP50-95": float(baseline_metrics.box.map),
}
(REPORT_DIR / "baseline_validation_summary.json").write_text(
    json.dumps(BASELINE_SUMMARY, indent=2), encoding="utf-8"
)
pd.DataFrame({
    "class_id": range(63),
    "class_name": EXPECTED_CLASSES,
    "baseline_mAP50-95": np.asarray(baseline_metrics.box.maps, dtype=float),
}).to_csv(REPORT_DIR / "baseline_validation_per_class.csv", index=False)
display(pd.DataFrame([BASELINE_SUMMARY]))


## 8. Fine-tune from the deployed Version 2 checkpoint

Only Train and Validation influence this run. Horizontal and vertical flips remain disabled because they change sign meaning.


In [ ]:
assert SPLIT_APPROVED and EXPECTED_SPLIT_IMAGES == ACTUAL_SPLIT_IMAGES
assert not RUN_DIR.exists(), f"Run folder already exists: {RUN_DIR}. Use the recovery cell after an interruption."
update_model = YOLO(str(BASE_MODEL_PATH), task="detect")
update_model.train(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    patience=PATIENCE, device=0, workers=WORKERS,
    project=str(RUN_ROOT), name=RUN_NAME, exist_ok=False,
    pretrained=True, resume=False, optimizer="AdamW",
    lr0=0.001, lrf=0.01, cos_lr=True, weight_decay=0.0005,
    cls_pw=CLASS_WEIGHT_POWER, cache="disk", amp=True, plots=True,
    save_period=5, seed=42, deterministic=True,
    hsv_h=0.015, hsv_s=0.50, hsv_v=0.30,
    degrees=8, translate=0.05, scale=0.25,
    fliplr=0.0, flipud=0.0, mosaic=0.50, close_mosaic=10,
)
BEST_PT = RUN_DIR / "weights" / "best.pt"
LAST_PT = RUN_DIR / "weights" / "last.pt"
assert BEST_PT.is_file(), f"Training did not create {BEST_PT}"
print("New best checkpoint:", BEST_PT)


## 9. Compare both models on Validation

Use this result and saved camera recordings to decide whether the update is acceptable. Do not open Test yet.


In [ ]:
import matplotlib.pyplot as plt

BEST_PT = RUN_DIR / "weights" / "best.pt"
assert BEST_PT.is_file(), f"Missing {BEST_PT}"
baseline_path = REPORT_DIR / "baseline_validation_summary.json"
baseline_class_path = REPORT_DIR / "baseline_validation_per_class.csv"
assert baseline_path.is_file() and baseline_class_path.is_file(), "Run the baseline cell first."
best_model = YOLO(str(BEST_PT), task="detect")
update_metrics = best_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=0, project=str(EVAL_ROOT),
    name=f"updated_model_rf_v{VERSION}_validation", exist_ok=True, plots=True,
)
UPDATE_VALIDATION_SUMMARY = {
    "model": f"Roboflow Version {VERSION} update",
    "precision": float(update_metrics.box.mp),
    "recall": float(update_metrics.box.mr),
    "mAP50": float(update_metrics.box.map50),
    "mAP50-95": float(update_metrics.box.map),
}
baseline_summary = json.loads(baseline_path.read_text(encoding="utf-8"))
comparison = pd.DataFrame([baseline_summary, UPDATE_VALIDATION_SUMMARY])
comparison.to_csv(REPORT_DIR / "validation_comparison.csv", index=False)
per_class = pd.read_csv(baseline_class_path)
per_class["update_mAP50-95"] = np.asarray(update_metrics.box.maps, dtype=float)
per_class["change"] = per_class["update_mAP50-95"] - per_class["baseline_mAP50-95"]
per_class.to_csv(REPORT_DIR / "validation_per_class_comparison.csv", index=False)
ax = comparison.set_index("model")[["precision", "recall", "mAP50", "mAP50-95"]].T.plot(
    kind="bar", figsize=(11, 6), color=["#94a3b8", "#2563eb"]
)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Deployed model vs annotated update on Validation")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(REPORT_DIR / "validation_comparison.png", dpi=200, bbox_inches="tight")
plt.show()
display(comparison)
print("Largest per-class decreases:")
display(per_class.sort_values("change").head(15))


## 10. Optional one-time final Test

Keep `RUN_FINAL_TEST = False` while any setting may still change. Set it to `True` only after the team accepts the model using Validation and freezes all choices.


In [ ]:
if not RUN_FINAL_TEST:
    print("Final Test skipped. This is correct while tuning remains possible.")
else:
    base_test = YOLO(str(BASE_MODEL_PATH), task="detect").val(
        data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
        device=0, project=str(EVAL_ROOT),
        name=f"deployed_model_rf_v{VERSION}_final_test", exist_ok=True, plots=True,
    )
    update_test = YOLO(str(BEST_PT), task="detect").val(
        data=str(DATA_YAML), split="test", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
        device=0, project=str(EVAL_ROOT),
        name=f"updated_model_rf_v{VERSION}_final_test", exist_ok=True, plots=True,
    )
    final_test_comparison = pd.DataFrame([
        {"model": "deployed Version 2", "precision": float(base_test.box.mp),
         "recall": float(base_test.box.mr), "mAP50": float(base_test.box.map50),
         "mAP50-95": float(base_test.box.map)},
        {"model": f"Roboflow Version {VERSION} update", "precision": float(update_test.box.mp),
         "recall": float(update_test.box.mr), "mAP50": float(update_test.box.map50),
         "mAP50-95": float(update_test.box.map)},
    ])
    final_test_comparison.to_csv(REPORT_DIR / "final_test_comparison.csv", index=False)
    display(final_test_comparison)


## 11. Export, verify OpenVINO parity, and create a Drive package

Run this only for the accepted checkpoint. The OpenVINO export is measured on Validation and must not lose more than 0.01 absolute mAP50 from its PyTorch source.


In [ ]:
from datetime import datetime
import shutil

BEST_PT = RUN_DIR / "weights" / "best.pt"
assert BEST_PT.is_file(), f"Missing accepted checkpoint: {BEST_PT}"
best_model = YOLO(str(BEST_PT), task="detect")
pytorch_export_metrics = best_model.val(
    data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device=0, project=str(EVAL_ROOT), name=f"accepted_pt_rf_v{VERSION}_validation",
    exist_ok=True, plots=False,
)
onnx_path = Path(best_model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True, end2end=True))
openvino_path = Path(best_model.export(format="openvino", imgsz=IMAGE_SIZE, end2end=True))
assert onnx_path.is_file(), f"ONNX export missing: {onnx_path}"
assert openvino_path.is_dir(), f"OpenVINO export missing: {openvino_path}"
openvino_metrics = YOLO(str(openvino_path), task="detect").val(
    data=str(DATA_YAML), split="val", imgsz=IMAGE_SIZE, batch=BATCH_SIZE,
    device="intel:cpu", project=str(EVAL_ROOT),
    name=f"accepted_openvino_rf_v{VERSION}_validation", exist_ok=True, plots=False,
)
map50_loss = float(pytorch_export_metrics.box.map50 - openvino_metrics.box.map50)
assert map50_loss <= 0.01, f"OpenVINO mAP50 loss is too large: {map50_loss:.4f}"

export_summary = {
    "pytorch_mAP50": float(pytorch_export_metrics.box.map50),
    "openvino_mAP50": float(openvino_metrics.box.map50),
    "absolute_mAP50_loss": map50_loss,
}
(REPORT_DIR / "export_validation_summary.json").write_text(
    json.dumps(export_summary, indent=2), encoding="utf-8"
)
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
PACKAGE_DIR = BACKUP_ROOT / f"{RUN_NAME}_{stamp}"
PACKAGE_DIR.mkdir(parents=True, exist_ok=False)
shutil.copytree(RUN_DIR, PACKAGE_DIR / "training_run")
shutil.copytree(REPORT_DIR, PACKAGE_DIR / "report_artifacts")
shutil.copytree(EVAL_ROOT, PACKAGE_DIR / "evaluations")
shutil.copy2(DATA_YAML, PACKAGE_DIR / "data.yaml")
manifest = {
    "project": "MYSignVoice",
    "roboflow_version": VERSION,
    "source_checkpoint": "deployed Version 2 best.pt",
    "best_checkpoint": "training_run/weights/best.pt",
    "classes": EXPECTED_CLASSES,
    "image_size": IMAGE_SIZE,
    "split_images": ACTUAL_SPLIT_IMAGES,
    "final_test_run": RUN_FINAL_TEST,
    "export_validation": export_summary,
}
(PACKAGE_DIR / "model_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)
LATEST_ARCHIVE = Path(shutil.make_archive(str(PACKAGE_DIR), "zip", root_dir=PACKAGE_DIR))
print("PASS: OpenVINO parity check passed.")
print("Package folder:", PACKAGE_DIR)
print("ZIP backup:", LATEST_ARCHIVE)
print("PyTorch model:", PACKAGE_DIR / "training_run" / "weights" / "best.pt")


## 12. Show a clickable ZIP link


In [ ]:
from IPython.display import FileLink, display
archives = sorted(BACKUP_ROOT.glob(f"{RUN_NAME}_*.zip"), key=lambda path: path.stat().st_mtime)
assert archives, "No ZIP exists. Run the export cell first."
LATEST_ARCHIVE = archives[-1]
print("ZIP size (GB):", round(LATEST_ARCHIVE.stat().st_size / 1024**3, 3))
display(FileLink(str(LATEST_ARCHIVE)))


## Optional recovery after an interrupted training cell

Use this only when Cell 8 started and was interrupted. Do not use it to extend an already completed run.


In [ ]:
LAST_PT = RUN_DIR / "weights" / "last.pt"
assert LAST_PT.is_file(), f"No interrupted checkpoint found: {LAST_PT}"
YOLO(str(LAST_PT), task="detect").train(resume=True)
